# Stage 4/5:RC Design + 強柱弱梁——第一次真正接上 `Stage 2/3` 的需求

**這裡的結果是「`Stage 4` 強度設計結果」,不是「最終施工配筋」**——
完整的耐震設計還需要正負彎矩要求、梁端連續筋、箍筋/圍束、`capacity
-design shear`、梁柱接頭、錨定搭接、間距等一系列 `Stage 5` 檢核都
通過之後,才能正式 `Design Freeze`。這一課只完成 `PM interaction`+
最小鋼筋比+`SCWB` 這幾項,不是完整清單。

`Case-04.7`(`Stage 2/3`)完成了四工具(`OpenSeesPy`/`frame2d`/`PyNite`/
`PyFEM`)驗證過的 `Canonical Elastic Model`,算出了真正可信的
`governing demand`。這一課第一次把這組需求接進正式的 `RC` 設計流程
——不是延續 `Case-06` 系列從頭到尾都沒變過的 $\rho=0.02$ 假設值,
是真正從需求反推出配筋。

**這不是延續 `case03_7`**——`case03_7` 用的是自己獨立的簡化剪力構架
需求(`Mu≈22.6`),不是 `Stage 2/3` 四工具驗證過的版本,兩者是不同
資料來源,不該混用。

## 第 1 課:柱配筋設計——用固定斷面搜尋最小滿足需求的配筋比

斷面(`40×40cm`)是 `Stage 2/3` 已確立的初步試設,這裡不重新搜尋
斷面尺寸,只搜尋配筋量。

In [1]:

import os
if not os.path.exists('rc_design.py'):
    !wget -q -O rc_design.py https://raw.githubusercontent.com/zhixiu0223/taiwan-seismic-code-calc/main/rc_design.py

from rc_design import design_column_PM, design_rebar

def Mn_at_Pu(curve, Pu_target):
    sorted_curve = sorted(curve, key=lambda p: p['Pn'])
    for i in range(len(sorted_curve)-1):
        if sorted_curve[i]['Pn'] <= Pu_target <= sorted_curve[i+1]['Pn']:
            t = (Pu_target-sorted_curve[i]['Pn'])/(sorted_curve[i+1]['Pn']-sorted_curve[i]['Pn'])
            return sorted_curve[i]['Mn'] + t*(sorted_curve[i+1]['Mn']-sorted_curve[i]['Mn'])
    return None

def design_column_rebar(b_cm, h_cm, Pu_kN, Mu_kNm, fc=280.0, fy=4200.0, Es=2.0e6,
                          cover_to_center=4.0, rho_candidates=None):
    """柱配筋設計迴圈: 給定固定斷面, 從真實需求反推出剛好滿足需求
    的最小配筋比——不是延續假設值。候選清單從ACI 318柱最小配筋比
    (1%)開始, 因為規範不允許低於這個值, 不是demand本身的限制。"""
    if rho_candidates is None:
        rho_candidates = [0.010, 0.012, 0.014, 0.016, 0.018, 0.020, 0.025, 0.030, 0.040]
    for rho in rho_candidates:
        As_bar = rho*b_cm*h_cm/8
        As_layers = [3*As_bar, 2*As_bar, 3*As_bar]
        r = design_column_PM(b_cm=b_cm, h_cm=h_cm, As_per_layer_cm2=As_layers,
                               cover_to_center_cm=cover_to_center, n_layers=3,
                               fc=fc, fy=fy, Es=Es, Pu_kN=Pu_kN, Mu_kNm=Mu_kNm)
        if r['within_envelope']:
            return dict(rho=rho, As_total=8*As_bar, r=r,
                        Mn=Mn_at_Pu(r['curve'], Pu_kN))
    return None

# governing demand來自Case-04.7(Stage 2/3, 四工具驗證過)
print("=== 1F柱(governing: Pu=308.85kN, Mu=32.13kN-m) ===")
result_1F = design_column_rebar(40.0, 40.0, 308.85, 32.13)
print(f"最小滿足需求的rho = {result_1F['rho']:.3%}")
print(f"As_total = {result_1F['As_total']:.2f}cm^2, Mn = {result_1F['Mn']:.2f}kN-m")
print(f"utilization = {result_1F['r']['utilization']:.4f}")

print()
print("=== 2F柱(governing: Pu=153.19kN, Mu=11.06kN-m) ===")
result_2F = design_column_rebar(40.0, 40.0, 153.19, 11.06)
print(f"最小滿足需求的rho = {result_2F['rho']:.3%}")
print(f"As_total = {result_2F['As_total']:.2f}cm^2, Mn = {result_2F['Mn']:.2f}kN-m")
print(f"utilization = {result_2F['r']['utilization']:.4f}")

# 確認demand本身允許的下限(避免候選清單邊界誤導判斷)
print()
print("=== 驗證: 最小配筋比是不是被規範鎖住, 不是被demand本身要求 ===")
for rho_test in [0.004, 0.006, 0.008]:
    As_bar_t = rho_test*40.0*40.0/8
    As_layers_t = [3*As_bar_t, 2*As_bar_t, 3*As_bar_t]
    r_t = design_column_PM(b_cm=40.0, h_cm=40.0, As_per_layer_cm2=As_layers_t,
                             cover_to_center_cm=4.0, n_layers=3,
                             fc=280.0, fy=4200.0, Es=2.0e6, Pu_kN=308.85, Mu_kNm=32.13)
    print(f"  rho={rho_test:.3%}: within_envelope={r_t['within_envelope']}, utilization={r_t['utilization']:.4f}")
print("即使rho=0.4%都還滿足需求——這代表真正governing的是ACI 318最小")
print("配筋比規範(1%), 不是強度需求本身; 這根柱斷面(40x40cm)相對真實")
print("需求來說偏大, 容量遠超實際需要")


=== 1F柱(governing: Pu=308.85kN, Mu=32.13kN-m) ===
最小滿足需求的rho = 1.000%
As_total = 16.00cm^2, Mn = 159.97kN-m
utilization = 0.2097

=== 2F柱(governing: Pu=153.19kN, Mu=11.06kN-m) ===
最小滿足需求的rho = 1.000%
As_total = 16.00cm^2, Mn = 136.83kN-m
utilization = 0.0818

=== 驗證: 最小配筋比是不是被規範鎖住, 不是被demand本身要求 ===
  rho=0.400%: within_envelope=True, utilization=0.2307
  rho=0.600%: within_envelope=True, utilization=0.2279
  rho=0.800%: within_envelope=True, utilization=0.2127
即使rho=0.4%都還滿足需求——這代表真正governing的是ACI 318最小
配筋比規範(1%), 不是強度需求本身; 這根柱斷面(40x40cm)相對真實
需求來說偏大, 容量遠超實際需要


## 第 2 課:梁配筋設計——同樣用 `Stage 2/3` 的真實需求

**誠實記錄一個重要釐清**:這裡算出的梁需求(`Mu≈24.19`/`16.78`)
遠低於 `VL-14` 當時反推的值(`Mu=428.44`)——**這不是矛盾,是兩者
代表完全不同的分析層級**:這裡用的是規範等效靜力法(設計地震力)
下的線彈性需求,是標準 `RC` 設計規範該用的層級;`VL-14` 用的是
`pushover` 側推分析推到 `4%` 層間位移(遠超設計地震力)時的極限
層級需求。標準 `RC` 設計只用規範地震力做配筋,不會用側推分析的
極限內力去決定該配多少鋼筋——`VL-14` 當時「從 `pushover` 結果反推
配筋」這個做法,事後看來混用了設計層級跟驗證層級的需求,兩者都有
各自正確的用途,不是誰對誰錯。

In [2]:

print("=== 1F樑(governing: Mu=24.19kN-m) ===")
r_beam_1F = design_rebar(24.19, 30.0, 50.0, cover=4.0)
print(f"rho_req(強度需求) = {r_beam_1F['rho_req']:.5f}")
print(f"rho_min(最小鋼筋比, ACI 318標準公式max(14/fy, 0.8*sqrt(fc')/fy)) = {r_beam_1F['rho_min']:.5f}")
print(f"governing: {'最小鋼筋比' if r_beam_1F['rho_min'] > r_beam_1F['rho_req'] else '強度需求'}")
print(f"As_provided={r_beam_1F['As_provided']:.2f}cm^2, phiMn={r_beam_1F['phiMn_provided']:.2f}")
print(f"bar_size={r_beam_1F['bar_size']}, n_bars={r_beam_1F['n_bars']}")

print()
print("=== 屋頂樑(governing: Mu=16.78kN-m) ===")
r_beam_roof = design_rebar(16.78, 30.0, 50.0, cover=4.0)
print(f"rho_req(強度需求) = {r_beam_roof['rho_req']:.5f}")
print(f"rho_min(最小鋼筋比) = {r_beam_roof['rho_min']:.5f}")
print(f"governing: {'最小鋼筋比' if r_beam_roof['rho_min'] > r_beam_roof['rho_req'] else '強度需求'}")
print(f"As_provided={r_beam_roof['As_provided']:.2f}cm^2, phiMn={r_beam_roof['phiMn_provided']:.2f}")
print(f"bar_size={r_beam_roof['bar_size']}, n_bars={r_beam_roof['n_bars']}")

print()
print("[誠實記錄] 兩根梁都被最小鋼筋比govern, 不是強度需求算出來的")
print("結果——這不是設計函式漏掉最小鋼筋比檢查, 是design_rebar()")
print("本身已經內建這個規範要求(rho_min), 只是上一版沒有明確呈現")
print("這層governing關係, 容易讓人誤以為只是單純強度算出的配筋")


=== 1F樑(governing: Mu=24.19kN-m) ===
rho_req(強度需求) = 0.00115
rho_min(最小鋼筋比, ACI 318標準公式max(14/fy, 0.8*sqrt(fc')/fy)) = 0.00333
governing: 最小鋼筋比
As_provided=5.73cm^2, phiMn=89.45
bar_size=#6(D19), n_bars=2

=== 屋頂樑(governing: Mu=16.78kN-m) ===
rho_req(強度需求) = 0.00079
rho_min(最小鋼筋比) = 0.00333
governing: 最小鋼筋比
As_provided=5.73cm^2, phiMn=89.45
bar_size=#6(D19), n_bars=2

[誠實記錄] 兩根梁都被最小鋼筋比govern, 不是強度需求算出來的
結果——這不是設計函式漏掉最小鋼筋比檢查, 是design_rebar()
本身已經內建這個規範要求(rho_min), 只是上一版沒有明確呈現
這層governing關係, 容易讓人誤以為只是單純強度算出的配筋


## 第 3 課:強柱弱梁檢核(`Stage 5`)——`ACI 318` 對特殊抗彎構架的規範要求

$\sum M_{nc} \geq 1.2\sum M_{nb}$,每個接頭都要檢核。用剛設計出的
柱(`1F`,`rho=1%`)跟梁(`1F`,`2-D19`)在 `1F` 接頭做示範。

In [3]:

Mn_col_1F = result_1F['Mn']
Mn_beam_1F = r_beam_1F['phiMn_provided']/0.9  # 反推標稱值(phi=0.9拉力控制)

# 1F接頭: 上下各一根柱匯入(1F柱跟2F柱), 這裡先用1F柱Mn做保守示範
sum_Mnc = 2*Mn_col_1F
sum_Mnb = Mn_beam_1F

ratio = sum_Mnc/sum_Mnb
print(f"柱標稱容量Mn = {Mn_col_1F:.2f}kN-m")
print(f"梁標稱容量Mn = {Mn_beam_1F:.2f}kN-m")
print(f"sum(Mnc) = {sum_Mnc:.2f}, 1.2*sum(Mnb) = {1.2*sum_Mnb:.2f}")
print(f"比值 = {ratio:.2f}(規範要求 >= 1.2)")

assert ratio >= 1.2, "強柱弱梁檢核應該通過"
print(f"\n[PASS] 強柱弱梁檢核通過, 餘裕{ratio:.2f}倍")
print()
print("對照Case-06.6(用假設rho=0.02柱+VL-14反推的超大梁配筋)算出")
print("\"柱先降伏\"(強梁弱柱)這個結果——這次用一致的規範需求層級")
print("設計出的柱梁配筋, 強柱弱梁大幅通過, 方向完全相反。這印證了")
print("Case-06.6那次結果建立在不一致demand層級混用之上, 不是規範")
print("設計流程該有的健康結果")


柱標稱容量Mn = 159.97kN-m
梁標稱容量Mn = 99.39kN-m
sum(Mnc) = 319.95, 1.2*sum(Mnb) = 119.27
比值 = 3.22(規範要求 >= 1.2)

[PASS] 強柱弱梁檢核通過, 餘裕3.22倍

對照Case-06.6(用假設rho=0.02柱+VL-14反推的超大梁配筋)算出
"柱先降伏"(強梁弱柱)這個結果——這次用一致的規範需求層級
設計出的柱梁配筋, 強柱弱梁大幅通過, 方向完全相反。這印證了
Case-06.6那次結果建立在不一致demand層級混用之上, 不是規範
設計流程該有的健康結果


## 第 4 課:梁的正負彎矩配筋要求(`ACI 318` 特殊抗彎構架規定)

耐震梁端承受反覆載重(正向地震跟反向地震方向相反),兩側(上緣/下緣)
都需要足夠的連續鋼筋,不能只做單向配筋設計。這裡從 `Case-04.7`
(`Stage 2/3`)重新抓出**帶正負號**的梁端彎矩(不是之前用的 `|M|`
包絡值),分別設計上緣(負彎矩)跟下緣(正彎矩)配筋。

**誠實記錄一個模型局限性**:`D+L`(純重力)情況下,這個模型算出的
梁端彎矩幾乎是 `0`——這是因為 `Stage 2/3` 的重力載重集中施加在
柱頂節點,沒有以分佈載重的形式直接作用在梁上,梁本身沒有承受真實
存在的「跨中正彎矩、支承處負彎矩」這種典型連續梁行為。這代表這
一課算出的正負彎矩需求,幾乎完全由側推力決定,剛好大小相等——
這是模型簡化造成的巧合,不是耐震設計的通例,拿掉「梁上真的有分佈
重力載重」這個簡化之後,正負彎矩不見得會再相等。

In [4]:

try:
    import openseespy.opensees as ops
except ImportError:
    import sys
    !{sys.executable} -m pip install -q openseespy
    import openseespy.opensees as ops

# 完全沿用Case-04.7(Stage 2/3)的幾何/材料/斷面參數, 這份notebook是
# 獨立檔案, 沒有繼承那邊的變數定義, 這裡重新宣告一次保持一致
L_bay = 6.0; h1 = 3.5; h2 = 3.5
h_col = 0.40; b_beam, h_beam = 0.30, 0.50
E_rc = 2.463e7
Ig_col = h_col**4/12; Ig_beam = b_beam*h_beam**3/12
Ic = 0.7*Ig_col; Ib = 0.35*Ig_beam
Ac = h_col**2; Ab = b_beam*h_beam
P_2F = 147.60; P_1F_extra = 147.60
F1_Y, F2_Y = 9.938, 15.900

def build():
    ops.wipe(); ops.model('basic', '-ndm', 2, '-ndf', 3)
    ops.node(1, 0.0, 0.0);   ops.node(2, L_bay, 0.0)
    ops.node(3, 0.0, h1);    ops.node(4, L_bay, h1)
    ops.node(5, 0.0, h1+h2); ops.node(6, L_bay, h1+h2)
    ops.fix(1,1,1,1); ops.fix(2,1,1,1)
    ops.geomTransf('Linear', 1)
    ops.element('elasticBeamColumn', 1, 1, 3, Ac, E_rc, Ic, 1)
    ops.element('elasticBeamColumn', 2, 2, 4, Ac, E_rc, Ic, 1)
    ops.element('elasticBeamColumn', 3, 3, 5, Ac, E_rc, Ic, 1)
    ops.element('elasticBeamColumn', 4, 4, 6, Ac, E_rc, Ic, 1)
    ops.element('elasticBeamColumn', 5, 3, 4, Ab, E_rc, Ib, 1)
    ops.element('elasticBeamColumn', 6, 5, 6, Ab, E_rc, Ib, 1)

def run_case(loads):
    build()
    ops.timeSeries('Linear', 1); ops.pattern('Plain', 1, 1)
    for (n, fx, fy, mz) in loads:
        ops.load(n, fx, fy, mz)
    ops.system('BandGeneral'); ops.numberer('Plain'); ops.constraints('Plain')
    ops.algorithm('Linear'); ops.integrator('LoadControl', 1.0); ops.analysis('Static')
    ops.analyze(1)
    return ops.eleForce(5), ops.eleForce(6)

f5_g, f6_g = run_case([(3,0,-P_1F_extra,0), (4,0,-P_1F_extra,0), (5,0,-P_2F,0), (6,0,-P_2F,0)])
f5_l, f6_l = run_case([(3,F1_Y,0,0), (5,F2_Y,0,0)])

Mi_DLE = f5_g[2]+f5_l[2]; Mj_DLE = f5_g[5]+f5_l[5]
Mi_DLnE = f5_g[2]-f5_l[2]; Mj_DLnE = f5_g[5]-f5_l[5]
print(f"1F樑 D+L+E: Mi={Mi_DLE:.3f}, Mj={Mj_DLE:.3f}")
print(f"1F樑 D+L-E: Mi={Mi_DLnE:.3f}, Mj={Mj_DLnE:.3f}")

# 上緣(負彎矩)取D+L+E那一組(Mi更負), 下緣(正彎矩)取D+L-E那一組
Mu_neg = abs(min(Mi_DLE, Mj_DLE))
Mu_pos = abs(max(Mi_DLnE, Mj_DLnE))

r_neg = design_rebar(Mu_neg, 30.0, 50.0, cover=4.0)
r_pos = design_rebar(Mu_pos, 30.0, 50.0, cover=4.0)
print(f"\n上緣(負彎矩, Mu={Mu_neg:.3f})配筋: As={r_neg['As_provided']:.2f}cm^2, {r_neg['bar_size']}x{r_neg['n_bars']}")
print(f"下緣(正彎矩, Mu={Mu_pos:.3f})配筋: As={r_pos['As_provided']:.2f}cm^2, {r_pos['bar_size']}x{r_pos['n_bars']}")

Mn_neg_1F = r_neg['phiMn_provided']/0.9
Mn_pos_1F = r_pos['phiMn_provided']/0.9
ratio_a = Mn_pos_1F/Mn_neg_1F
print(f"\n[ACI 318檢核a] Mn+/Mn- = {ratio_a:.3f}(規範要求 >= 0.5) {'PASS' if ratio_a >= 0.5 else 'FAIL'}")

# 檢核b: 任意位置容量 >= 0.25*max(接頭面容量)——這裡假設全長同一組
# 配筋(彈性均質梁模型沒有沿長度變化配筋這個細節), 自動滿足
max_Mn_face = max(Mn_neg_1F, Mn_pos_1F)
print(f"[ACI 318檢核b] 假設全長同一組配筋, 任意位置容量={min(Mn_neg_1F,Mn_pos_1F):.2f} "
      f">= 0.25*{max_Mn_face:.2f}={0.25*max_Mn_face:.2f}? "
      f"{'PASS' if min(Mn_neg_1F,Mn_pos_1F) >= 0.25*max_Mn_face else 'FAIL'}")
print("(這個檢核在\"全長不變配筋\"這個簡化下必然通過, 真正有意義的")
print(" 情境是梁中央有意減少鋼筋量節省成本時, 這個模型還沒有處理這件事)")


1F樑 D+L+E: Mi=-24.189, Mj=-24.168
1F樑 D+L-E: Mi=24.189, Mj=24.168

上緣(負彎矩, Mu=24.189)配筋: As=5.73cm^2, #6(D19)x2
下緣(正彎矩, Mu=24.189)配筋: As=5.73cm^2, #6(D19)x2

[ACI 318檢核a] Mn+/Mn- = 1.000(規範要求 >= 0.5) PASS
[ACI 318檢核b] 假設全長同一組配筋, 任意位置容量=99.39 >= 0.25*99.39=24.85? PASS
(這個檢核在"全長不變配筋"這個簡化下必然通過, 真正有意義的
 情境是梁中央有意減少鋼筋量節省成本時, 這個模型還沒有處理這件事)


## 小結:`Design Freeze` 前的完整規範閉環

- 柱:用 `Stage 2/3` 四工具驗證過的真實需求反推,發現 $\rho=0.02$
  這個從 `Case-06` 系列一開始就存在的假設值,遠遠超過實際需要
  (`utilization` 只有 `0.08~0.21`),真正 governing 的是 `ACI 318`
  最小配筋比規範(`1%`),不是強度需求
- 梁:同樣用 `Stage 2/3` 的規範等效靜力法需求設計,配筋遠比 `VL-14`
  當時反推的輕量——這不是矛盾,是設計層級(規範地震力)跟驗證層級
  (`pushover` 極限需求)本來就不該混用同一組配筋反推邏輯
- 強柱弱梁:用這次一致需求層級設計出的柱梁配筋,大幅通過(餘裕
  `3.22` 倍),跟 `Case-06.6` 那次「柱先降伏」的結果方向相反——
  印證了 `demand` 層級一致性的重要性
- 這是 `Design Freeze` 前的關鍵一步:確認柱梁配筋、強柱弱梁都通過
  之後,才能真正凍結這組設計結果,餵給後續的非線性模型